In [2]:
import re
import os
from typing import Optional
from datetime import datetime
from dataclasses import dataclass, field

In [12]:
@dataclass
class GaurdrailResult:
    passed :bool
    layer :str
    reason :str
    score: Optional[float] = None
    redacted : Optional[str] = None
    timestamp :str = field(default_factory = lambda: datetime.utcnow().isoformat())

    def __str__(self):
        status = "PASS" if self.passed else "BLOCK"
        score = f"score = {self.score:.3f}"  if self.score is not None else "" 
        return f"{self.layer} -  {status} {score} : {self.reason}"
    





In [19]:
## 
TEXT_RISK = ["Harmful text output","PII data leaked","Jailbreak to harmful content"]


AGENTIC_RISK = ["Unauthrized tool calls (deleting database when agent has only read access",
"parameter injection (99999 which causes masseive data export)",
"injection in the tool resut ",
"Agents runs the tools call in loop",
"Budget will be increase",
"Privalage based access"]


In [29]:
TOOL_REGISTERY :dict[str,str] = {
    "customer_service":{"search_faq","get_order_status","send_confirmation_email"},
    "data_analyst":{"query_warehouse","generate_chart","export_csv"},
    "code_assistant":{"run_test","lint_code","search_documentation"},
    "admin":{"create_user","delete_user","modify_permission","export_csv"}
}


TOOL_SCHEMAS : dict[str,str] = {

    "get_order_status":{"order_id":str},
    "send_confirmation_email":{"to_email":str, "template_id":str},
    "query_warehouse":{"sql":str,"limit":int,"dry_run":bool},
    "export_csv":{"table":str,"num_rows":int},
    "run_test":{"test_path":str,"timeout_seconds":int}
}

In [30]:
@dataclass
class AgentSession:
    session_id: str
    role: str
    total_input_tokens: int  = 0
    total_output_tokens: int = 0
    tool_calls: list         = field(default_factory=list)
    spend_usd: float         = 0.0

    SESSION_BUDGET_USD      = 0.50
    COST_PER_1K_INPUT       = 0.0005
    COST_PER_1K_OUTPUT      = 0.0015

    def record_usage(self, input_tokens: int, output_tokens: int):
        self.total_input_tokens  += input_tokens
        self.total_output_tokens += output_tokens
        self.spend_usd = (
            (self.total_input_tokens  / 1000) * self.COST_PER_1K_INPUT +
            (self.total_output_tokens / 1000) * self.COST_PER_1K_OUTPUT
        )

    def budget_remaining(self) -> float:
        return max(0.0, self.SESSION_BUDGET_USD - self.spend_usd)

print("AgentSession defined ✅")

AgentSession defined ✅


In [39]:
def validate_tool_call(
    session: AgentSession,
    tool_name: str,
    params: dict,
) -> GaurdrailResult:
    """
    Layer 5a: two-level validation.
    (1) Is the tool whitelisted for this agent's role?
    (2) Are the parameters well-typed according to the schema?
    """
    allowed = TOOL_REGISTERY.get(session.role, set())

    # Check 1: whitelist
    if tool_name not in allowed:
        return GaurdrailResult(
            passed=False, layer="L5-ToolWhitelist",
            reason=f"Tool '{tool_name}' not in whitelist for role '{session.role}'. "
                   f"Allowed: {', '.join(sorted(allowed))}",
        )

    # Check 2: parameter schema
    schema = TOOL_SCHEMAS.get(tool_name, {})
    for param_name, expected_type in schema.items():
        if param_name  not in params :
            return GaurdrailResult(
                passed=False, layer="L5-ParamValidation",
                reason=f"Missing required Parameter {param_name}",
            )
        if param_name in params and not isinstance(params[param_name], expected_type):
            return GaurdrailResult(
                passed=False, layer="L5-ParamValidation",
                reason=f"'{param_name}' must be {expected_type.__name__}, "
                       f"got {type(params[param_name]).__name__}",
            )

    # Check 3: reasonable limits
    if "limit" in params and isinstance(params.get("limit"), int) and params["limit"] > 10_000:
        return GaurdrailResult(
            passed=False, layer="L5-ParamValidation",
            reason=f"Parameter 'limit'={params['limit']} exceeds maximum safe value of 10,000",
        )
    if "max_rows" in params and isinstance(params.get("max_rows"), int) and params["max_rows"] > 50_000:
        return GaurdrailResult(
            passed=False, layer="L5-ParamValidation",
            reason=f"Parameter 'max_rows'={params['max_rows']} exceeds maximum of 50,000",
        )

    session.tool_calls.append(tool_name)
    return GaurdrailResult(
        passed = True, layer = "L5-ToolWhitelist", reason = f"Tool {tool_name} approved for role '{session.role}'"
    )

In [ ]:
session_cs = AgentSession(session_id = "cs-001",role = "customer_service")

test_call = [(session_cs,"export_csv","","approved")]

for session,tool,params,expected in test_call:
    r = validate_tool_call(session,tool,params)
    allow = "ALLOWED" if r.passed else "BLOCKED"
    print(allow)

BLOCKED


/var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/ipykernel_11200/1689136956.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp :str = field(default_factory = lambda: datetime.utcnow().isoformat())


In [42]:
session_ds = AgentSession(session_id = "ds-001",role = "data_analyst")

test_call = [(session_ds,"export_csv",{"table":"employee","num_rows":50},"approved")]

for session,tool,params,expected in test_call:
    r = validate_tool_call(session,tool,params)
    allow = "ALLOWED" if r.passed else "BLOCKED"
    print(allow)

ALLOWED


/var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/ipykernel_11200/1689136956.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp :str = field(default_factory = lambda: datetime.utcnow().isoformat())
